In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
df_main = pd.read_excel("MentorMe-CV-File-20260224.xlsx",sheet_name="Main_table")
df_user = pd.read_csv("dim_user.csv")

In [10]:
df_main['candidate_name'] = df_main['candidate_name'].str.strip().str.lower()
df_main['email'] = df_main['email'].str.strip().str.lower()

In [11]:
df_merge = df_main.merge(df_user[['customer_id','candidate_name','email']],
                         on=['candidate_name','email'],
                         how='left')

df_merge.head(3).T

,0,1,2
id,2,3,5
user_id,NaN,NaN,NaN
analysis_id,1759998111377,1759998769335,1759999340602
candidate_name,christopher davis,minh vu,tran quoc bao
email,chris.davis.1990@email.com,vddminh89@gmail.com,21bao.tq@vinuni.edu.vn
phone,(415) 555-0776,(84) 090-4562633,(+84) 382780002
status,craft,craft,craft
file_name,Christopher Davis_Flutter Developer.pdf,CV_Minh Vu - Minh Vu.pdf,BaoTran_resume - BaÌo TraÌÌn.pdf
created_at,2025-10-09T08:22:24,2025-10-09T08:33:36,2025-10-09T08:42:56
updated_at,2025-10-09T08:22:24,2025-10-09T08:33:36,2025-10-09T08:42:56


In [12]:
df_work = df_merge[['customer_id','analysis_id','status','job_role','experience_level.level','experience_level.years','overview','strengths','gaps','recommendations','keywords','formatting_issues','regional_insights','created_at','file_name']].copy()
df_work.head(2).T

,0,1
customer_id,1,2
analysis_id,1759998111377,1759998769335
status,craft,craft
job_role,Flutter development,Data analytics
experience_level.level,Mid level,Senior level
experience_level.years,2,9
overview,Proficient Flutter Developer with 2 years buil...,Seasoned System Manager with 9 years of ERP an...
strengths,"[""Proficient Flutter & Dart development"",""Cros...","[""ERP systems implementation with Odoo"",""BI re..."
gaps,"[""Generic project description"",""Missing GitHub...","[""Missing professional summary section"",""Lacks..."
recommendations,"[""Add actual GitHub project links"",""Quantify a...","[""Add a professional summary highlighting key ..."


In [13]:
# As we can see these are somes columns have json list we need to work on
# We will try with one column first

raw = df_work['gaps'].iloc[0]
parsed = json.loads(raw)
print(parsed)
print(type(parsed))

['Generic project description', 'Missing GitHub link URL', 'No quantified achievements', 'Lacks professional certifications', 'Incomplete project details']
<class 'list'>


In [18]:
# We make a function to flaten the list
def flatten_list(val):
    if pd.isnull(val):
        None
    parsed = json.loads(val)
    return ' | '.join(parsed)

In [23]:
# apply it and create new columns for our dataset
json_cols = ['strengths', 'gaps', 'recommendations', 
             'keywords', 'formatting_issues', 'regional_insights']

for k in json_cols:
    if k in df_work.columns:
        df_work[k] = df_work[k].apply(flatten_list)

In [24]:
df_work.head().T

,0,1,2,3,4
customer_id,1,2,3,4,4
analysis_id,1759998111377,1759998769335,1759999340602,1760001071242,1760001127938
status,craft,craft,craft,craft,craft
job_role,Flutter development,Data analytics,Artificial intelligence,Business analyst,Business analyst
experience_level.level,Mid level,Senior level,Entry level,Executive level,Senior level
experience_level.years,2,9,3,18,18
overview,Proficient Flutter Developer with 2 years buil...,Seasoned System Manager with 9 years of ERP an...,Electrical Engineering student with research e...,Project Manager with over 18 years in marketin...,Project and program management professional wi...
strengths,Proficient Flutter & Dart development | Cross-...,ERP systems implementation with Odoo | BI repo...,Expertise in robust EV charging optimization |...,Entrepreneurial leadership | Cross-continental...,Extensive program & project management experti...
gaps,Generic project description | Missing GitHub l...,Missing professional summary section | Lacks L...,No professional summary or objective statement...,No business analysis keywords | Lacks quantifi...,No quantifiable impact metrics | Missing busin...
recommendations,Add actual GitHub project links | Quantify ach...,Add a professional summary highlighting key ac...,Add a professional summary highlighting AI/ML ...,"Quantify achievements (e.g., % growth, budget ...","Include quantifiable achievements (e.g., reven..."


In [25]:
# Shape correct?
print(df_work.shape)

# No unexpected nulls in key columns?
print(df_work[['customer_id', 'analysis_id']].isnull().sum())

# JSON columns look clean?
print(df_work[['strengths', 'gaps', 'keywords']].head(3).T)

# Grain check — every analysis_id unique?
print(df_work['analysis_id'].nunique())

(191, 21)
customer_id    0
analysis_id    0
dtype: int64
                                                           0  \
strengths  Proficient Flutter & Dart development | Cross-...   
gaps       Generic project description | Missing GitHub l...   
keywords    Flutter | Dart | Firebase | RESTful APIs | Agile   

                                                           1  \
strengths  ERP systems implementation with Odoo | BI repo...   
gaps       Missing professional summary section | Lacks L...   
keywords   ERP | Odoo | SQL | Power BI | BI reporting | D...   

                                                           2  
strengths  Expertise in robust EV charging optimization |...  
gaps       No professional summary or objective statement...  
keywords   Python | Matlab | TensorFlow | Deep Learning |...  
189


In [27]:
df_work.duplicated().sum()

np.int64(0)

In [29]:
fact_cv_analysis = df_work[[
    'customer_id',
    'analysis_id', 
    'status',
    'job_role',
    'experience_level.level',
    'experience_level.years',
    'overview',
    'strengths',
    'gaps',
    'recommendations',
    'keywords',
    'formatting_issues',
    'regional_insights',
    'created_at',
    'file_name'
]]

fact_cv_analysis.to_csv('fact_cv_analysis.csv', index=False)
print("Done!", fact_cv_analysis.shape)

Done! (191, 15)
